# Earnalism edit job `selfish-giant-edit-test2-film-01`

**Prepared, not run.** Input zip SHA256 `c8133aa6e176157e86604611e1bd75e61a228102cff90fec15dee2dd55ce5dbb`. Set `RUN = True` in the first cell, then run all. With the R2 exchange (`exchange.json` in `/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-edit-test2-film-01/`) the notebook downloads `inputs.zip` itself and uploads the outputs when it finishes; without it, copy `inputs.zip` into that Drive folder first. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 2783 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'selfish-giant-edit-test2-film-01'
TITLE = 'the-selfish-giant'
KIND = 'edit'
INPUT_ZIP_SHA256 = 'c8133aa6e176157e86604611e1bd75e61a228102cff90fec15dee2dd55ce5dbb'
MODEL_REPO = 'Qwen/Qwen-Image-Edit-2511'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 40
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/the-selfish-giant/selfish-giant-edit-test2-film-01'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 2783
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'safetensors', 'bitsandbytes', 'torch', 'huggingface_hub')
EXCHANGE_SCHEMA = 'earnalism.colab-exchange.v1'
# The GPU cell stops before any download below this much GPU memory (0: any GPU)
MIN_VRAM_GIB = 30
LICENCE_ALLOWLIST = ('apache-2.0',)
LICENCE_REPOS = (MODEL_REPO,)
EDIT_PIPELINE = 'QwenImageEditPlusPipeline'
DIFFUSERS_PIN = 'diffusers==0.37.0'
# transformer bf16 from BF16_FROM_GIB of VRAM, 8-bit from INT8_FROM_GIB, else NF4;
# quantized loads keep SKIP_MODULES bf16
BF16_FROM_GIB = 70
INT8_FROM_GIB = 30
SKIP_MODULES = ['img_in', 'txt_in', 'time_text_embed', 'norm_out', 'proj_out']


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import shutil, subprocess
# A GPU first: a CPU-only runtime would install, download and then fail with a confusing CUDA error.
_smi = shutil.which("nvidia-smi")
if not _smi or subprocess.run([_smi, "-L"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU attached: Runtime -> Disconnect and delete runtime, then Runtime -> "
                       "Change runtime type -> A100 GPU (or L4), Save, Connect, and Run all again")
print(subprocess.run([_smi, "-L"], capture_output=True, text=True).stdout.strip())
_min_gib = globals().get("MIN_VRAM_GIB", 0)
if _min_gib:  # the job's spec min_vram_gib: refuse a smaller GPU before any install or download
    _q = subprocess.run([_smi, "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True)
    try:
        _gib = int(_q.stdout.split()[0]) / 1024  # MiB of the first GPU
    except (IndexError, ValueError):
        raise RuntimeError("could not read the GPU memory from nvidia-smi (" + repr(_q.stdout)
                           + "); this job needs at least %d GiB" % _min_gib) from None
    print("GPU memory %.1f GiB (this job needs at least %d GiB)" % (_gib, _min_gib))
    if _gib < _min_gib:
        raise RuntimeError("This job needs at least %d GiB of GPU memory and this GPU has %.1f GiB: it "
                           "needs an A100 or larger: Runtime -> Change runtime type -> A100, Save, "
                           "Connect, and Run all again (nothing was downloaded)" % (_min_gib, _gib))


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
_token, _secret_problem = None, None
try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN") or None
except Exception as _e:  # no secret, notebook access not granted, or the prompt timed out
    _secret_problem = type(_e).__name__
    _token = os.environ.get("HF_TOKEN") or None
if _token:
    os.environ["HF_TOKEN"] = _token
from huggingface_hub import HfApi
try:
    HfApi().auth_check(MODEL_REPO, token=_token)
except Exception as _e:
    _fix = ("accept the model's terms at https://huggingface.co/" + MODEL_REPO + " with your Hugging "
            "Face account, put a Read token from https://huggingface.co/settings/tokens in the Colab "
            "secret HF_TOKEN (key icon in the left sidebar, Notebook access on), then Runtime > "
            "Restart session and Run all")
    if _secret_problem:
        _fix += " (the HF_TOKEN secret could not be read: %s; grant access if Colab asks)" % _secret_problem
    raise RuntimeError("No access to " + MODEL_REPO + " (" + type(_e).__name__ + "): " + _fix) from None
print("model access OK:", MODEL_REPO, "(token from HF_TOKEN)" if _token else "(public)")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os
from huggingface_hub import HfApi
LICENCES = {}
for _repo in LICENCE_REPOS:
    try:
        HfApi().auth_check(_repo, token=os.environ.get("HF_TOKEN"))
        _info = HfApi().model_info(_repo, token=os.environ.get("HF_TOKEN"))
    except Exception as _e:
        raise RuntimeError("No access to " + _repo + " (" + type(_e).__name__ + "); nothing ran") from None
    _card = getattr(_info, "card_data", None) or getattr(_info, "cardData", None) or {}
    _lic = _card.get("license") if hasattr(_card, "get") else getattr(_card, "license", None)
    if not _lic:
        _lic = next((t.split(":", 1)[1] for t in (getattr(_info, "tags", None) or [])
                     if t.startswith("license:")), None)
    _lic = str(_lic).strip().lower() if _lic else None
    print("model card licence:", _repo, "->", _lic, "(allowlist: %s)" % ", ".join(LICENCE_ALLOWLIST))
    if _lic not in LICENCE_ALLOWLIST:
        raise RuntimeError("model card licence of " + _repo + " is " + repr(_lic) + ", not one of "
                           + ", ".join(LICENCE_ALLOWLIST) + ": this job never runs it; tell Claude")
    LICENCES[_repo] = _lic


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import importlib, inspect, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", DIFFUSERS_PIN, "transformers",
                "accelerate", "safetensors", "bitsandbytes"], check=True)
importlib.invalidate_caches()
import diffusers
print("diffusers", diffusers.__version__, "(pin: " + DIFFUSERS_PIN + ")")
if not hasattr(diffusers, EDIT_PIPELINE):
    raise RuntimeError("diffusers " + diffusers.__version__ + " has no " + EDIT_PIPELINE + "; install "
                       + DIFFUSERS_PIN + " (Runtime > Restart session, Run all); nothing downloaded")
if "zero_cond_t" not in inspect.signature(diffusers.QwenImageTransformer2DModel.__init__).parameters:
    raise RuntimeError("diffusers " + diffusers.__version__ + " QwenImageTransformer2DModel lacks "
                       "zero_cond_t (needed by " + MODEL_REPO + "); install " + DIFFUSERS_PIN)
print("pipeline OK:", EDIT_PIPELINE)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import threading, time
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, runtime.unassign)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    """Between items: start the next one only if it (and the manifest) fits before the hard stop."""
    global INCOMPLETE
    left = DEADLINE_SECONDS - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s forecast; no further item starts" % (left, DEADLINE_SECONDS))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
# R2 exchange (references/colab-exchange.md): exchange.json in this Drive folder carries two
# short-lived single-object URLs (GET inputs.zip, PUT outputs.tar). They are capabilities: never
# print them, never copy them elsewhere.
EXCHANGE = None
if (OUT_DIR / "exchange.json").is_file():
    import time, urllib.request
    from datetime import datetime
    EXCHANGE = json.loads((OUT_DIR / "exchange.json").read_text())
    if EXCHANGE.get("schema") != EXCHANGE_SCHEMA or EXCHANGE.get("job_id") != JOB_ID:
        raise RuntimeError("exchange.json belongs to a different job; ask Claude for this job's exchange.json")
    _left = datetime.fromisoformat(EXCHANGE["expires_at"].replace("Z", "+00:00")).timestamp() - time.time()
    if _left < DEADLINE_SECONDS + MANIFEST_RESERVE_S + 300:
        raise RuntimeError("the exchange URLs expire before this job's hard stop (%.0f s left); "
                           "ask Claude to re-sign them before running" % _left)
    if not INPUT_ZIP.is_file():
        with urllib.request.urlopen(EXCHANGE["inputs"]["url"], timeout=600) as _r:
            if not 200 <= _r.status < 300:
                raise RuntimeError("inputs.zip download failed: HTTP %d" % _r.status)
            _data = _r.read()
        _sha = hashlib.sha256(_data).hexdigest()
        if _sha != INPUT_ZIP_SHA256 or _sha != EXCHANGE["inputs"]["sha256"]:
            raise RuntimeError("downloaded inputs.zip sha256 mismatch: " + _sha + " (not kept)")
        INPUT_ZIP.write_bytes(_data)
        print("inputs.zip downloaded from the exchange:", len(_data), "bytes")
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)


In [ ]:
# Edit: instruction-following repair of a chosen still (Qwen-Image-Edit-2511, Apache-2.0).
# The still to edit goes first, then up to two reference images; the exact revision is resolved
# now and recorded. The licence and pipeline preflights already ran (pinned diffusers installed).
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import os, gc, hashlib, io, logging, math, warnings, zipfile
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


# --- helpers (pure PIL; unit-tested by the skill)
import math
from PIL import Image


def precision_for(vram_gib, bf16_from_gib, int8_from_gib):
    """The transformer's precision by total VRAM: bf16 from bf16_from_gib (A100 80 GB), 8-bit
    LLM.int8 from int8_from_gib (A100 40 GB), else 4-bit NF4 (L4). NF4 everywhere came out grainy."""
    if vram_gib >= bf16_from_gib:
        return "bf16"
    return "int8" if vram_gib >= int8_from_gib else "nf4"


def kept_bf16(precision, skip_modules):
    """The modules a quantized load keeps bf16 (none to keep when nothing is quantized)."""
    return [] if precision == "bf16" else list(skip_modules)


def edit_size(width, height, area=1024 * 1024, multiple=32):
    """The model's working size: about one megapixel at the source's aspect ratio, in multiples of
    32 (passed explicitly: the pipeline would otherwise follow the LAST input image, a reference)."""
    ratio = width / height
    w = max(multiple, int(round(math.sqrt(area * ratio) / multiple)) * multiple)
    h = max(multiple, int(round(math.sqrt(area / ratio) / multiple)) * multiple)
    return w, h


def fit_back(image, size):
    """The model output resized back to exactly the source still's size (RGB)."""
    image = image.convert("RGB")
    return image if image.size == tuple(size) else image.resize(tuple(size), Image.LANCZOS)
# --- end helpers


import torch
from diffusers import BitsAndBytesConfig, QwenImageEditPlusPipeline, QwenImageTransformer2DModel
from huggingface_hub import HfApi

revision = HfApi().model_info(MODEL_REPO).sha
gpu = torch.cuda.get_device_properties(0)
VRAM_GIB = gpu.total_memory / 2**30
PRECISION = precision_for(VRAM_GIB, BF16_FROM_GIB, INT8_FROM_GIB)
SKIPPED = kept_bf16(PRECISION, SKIP_MODULES)
print("GPU", gpu.name, "%.1f GiB;" % VRAM_GIB, "transformer", PRECISION,
      "; text encoder bf16; VAE decode float32; model CPU offload (one large part on the GPU at a time)")
if SKIPPED:
    print("kept in bf16 (not quantized):", ", ".join(SKIPPED))
if PRECISION != "bf16":
    # bitsandbytes logs a dtype-cast notice on every quantized matmul (thousands of lines per
    # image). Say it once here, then silence that logger and warning.
    logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", message=".*Input type into Linear4bit.*")
    warnings.filterwarnings("ignore", message=".*will be cast from.*", module="bitsandbytes")
    print(PRECISION, "transformer: bitsandbytes casts its inputs (cast notices shown once)")

# Sources and references: every PNG is checked against the sha256 the production record holds.
IMAGES = {}
with zipfile.ZipFile(INPUT_ZIP) as z:
    for item in JOB["items"]:
        wanted = [(item["source"], item["source_sha256"])] + [(r["member"], r["sha256"]) for r in item["refs"]]
        for member, digest in wanted:
            if member in IMAGES:
                continue
            data = z.read(member)
            if hashlib.sha256(data).hexdigest() != digest:
                raise RuntimeError("input image sha256 mismatch: " + member)
            IMAGES[member] = Image.open(io.BytesIO(data)).convert("RGB")

kw = {"revision": revision, "torch_dtype": torch.bfloat16}
if PRECISION == "int8":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=SKIPPED)
elif PRECISION == "nf4":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.bfloat16,
                                                   llm_int8_skip_modules=SKIPPED)
transformer = QwenImageTransformer2DModel.from_pretrained(MODEL_REPO, subfolder="transformer", **kw)
pipe = QwenImageEditPlusPipeline.from_pretrained(MODEL_REPO, revision=revision, transformer=transformer,
                                                 torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)


def decode_f32(latents, width, height):
    """The pipeline's own decode at diffusers 0.37.0 (unpack, de-normalise, decode the first frame),
    with the VAE in float32 to rule out bf16 decode banding. Only the decode is upcast: the
    pipeline feeds the VAE encoder bf16 inputs, so the VAE goes back to bf16 afterwards."""
    # dtype changes outside inference mode, so the VAE weights stay ordinary tensors
    pipe.vae.to(torch.float32)
    try:
        with torch.inference_mode():
            lat = pipe._unpack_latents(latents, height, width, pipe.vae_scale_factor)
            cfg = pipe.vae.config
            lat = lat.to(pipe._execution_device, torch.float32)
            mean = torch.tensor(cfg.latents_mean).view(1, cfg.z_dim, 1, 1, 1).to(lat.device, lat.dtype)
            inv_std = 1.0 / torch.tensor(cfg.latents_std).view(1, cfg.z_dim, 1, 1, 1).to(lat.device,
                                                                                         lat.dtype)
            image = pipe.vae.decode(lat / inv_std + mean, return_dict=False)[0][:, :, 0]
            return pipe.image_processor.postprocess(image, output_type="pil")[0]
    finally:
        pipe.vae.to(torch.bfloat16)
        pipe.maybe_free_model_hooks()  # offload the VAE again


import diffusers
SECONDS = {}
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "pipeline": EDIT_PIPELINE,
              "diffusers": diffusers.__version__, "diffusers_pin": DIFFUSERS_PIN,
              "licences": LICENCES, "gpu": gpu.name, "vram_gib": round(VRAM_GIB, 1),
              "transformer_precision": PRECISION, "transformer_kept_bf16": SKIPPED,
              "text_encoder_precision": "bf16", "vae_decode": "float32",
              "offload": "enable_model_cpu_offload", "output_size": "source size (LANCZOS)",
              "seconds_per_image": SECONDS}
EXPECTED = [i["key"] + ".png" for i in JOB["items"]]
last = 0.0  # seconds the previous candidate took; the next starts only if it fits (R25)
for item in JOB["items"]:
    if not may_start(last):
        break
    started = time.time()
    src = IMAGES[item["source"]]
    width, height = edit_size(*src.size)
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    with torch.inference_mode():
        latents = pipe(image=[src] + [IMAGES[r["member"]] for r in item["refs"]],
                       prompt=item["instruction"], negative_prompt=item["negative"],
                       true_cfg_scale=item["true_cfg_scale"], guidance_scale=1.0,
                       num_inference_steps=item["steps"], width=width, height=height,
                       num_images_per_prompt=1, generator=gen, output_type="latent").images
    out = decode_f32(latents, width, height)
    fit_back(out, src.size).save(OUT_DIR / (item["key"] + ".png"))
    last = time.time() - started
    SECONDS[item["key"]] = round(last, 1)
    print("wrote", item["key"], "%dx%d" % src.size, "(model %dx%d, %d steps)" % (width, height, item["steps"]),
          "in %.1f s per image" % last)
if SECONDS:
    print("seconds per image: mean %.1f, max %.1f over %d images (forecast: rates.json edit s_per_step x steps)"
          % (sum(SECONDS.values()) / len(SECONDS), max(SECONDS.values()), len(SECONDS)))
del pipe, transformer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if (p.is_file() and p.name not in ("manifest.json", "inputs.zip", "exchange.json")
            and p.suffix != ".ipynb"):  # Colab keeps the job notebook in this folder
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(time.time() - JOB_STARTED, 1),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
if globals().get("EXCHANGE") is not None:  # R2 exchange: one uncompressed tar, PUT once
    import os, tarfile, tempfile, urllib.error, urllib.request
    _fd, _tar = tempfile.mkstemp(suffix=".tar")  # local disk, never the Drive job folder
    os.close(_fd)
    with tarfile.open(_tar, "w", format=tarfile.PAX_FORMAT) as _t:
        for _name in ["manifest.json"] + sorted(outputs):
            _t.add(str(OUT_DIR / _name), arcname=_name, recursive=False)
    _size = os.path.getsize(_tar)
    _headers = dict(EXCHANGE["outputs"]["headers"], **{"Content-Length": str(_size)})
    with open(_tar, "rb") as _body:
        _req = urllib.request.Request(EXCHANGE["outputs"]["url"], data=_body, method="PUT",
                                      headers=_headers)
        try:
            with urllib.request.urlopen(_req, timeout=1800) as _r:
                _status = _r.status
        except urllib.error.HTTPError as _e:
            _status = _e.code
    os.unlink(_tar)
    print("outputs.tar upload: HTTP", _status, "(%d bytes)" % _size)
    if not 200 <= _status < 300:
        raise RuntimeError("outputs.tar upload failed: HTTP %d; the outputs stay in Drive, tell Claude"
                           % _status)
    print("outputs uploaded to the exchange; tell Claude the CU Colab shows")
